# Day 1 실습 — ChatPromptTemplate·LCEL 체인과 프롬프트 설계

**목표**: 프롬프트 양식을 만들어 체인으로 연결하고, 역할·지시문·맥락·예시를 하나씩 더하며 답 품질 변화를 직접 확인한다.
**구성**: Part 1 첫 체인 완성·오류 다루기 → Part 2 설계 요소 실험(+다른 도메인) → Part 3 미니 프로젝트(+나만의 캐릭터 챗봇)

> **참고:** 실습 전 가상환경 활성화, `.env`의 OpenAI API 키, 패키지 설치(`uv sync`)를 확인한다.

## 0. 환경 준비·모델 생성

필요한 도구를 불러오고 모델·파서 객체를 만든다. 이 셀은 노트북 전체에서 한 번만 실행한다.

<details>
<summary>각 import의 역할</summary>

- `ChatOpenAI`: OpenAI 채팅 모델을 부르는 객체다.
- `ChatPromptTemplate`: 프롬프트 양식을 만드는 도구다.
- `StrOutputParser`: 답을 순수 문자열로 정리하는 파서다.
- `load_dotenv`: `.env`의 API 키를 불러온다.
</details>

In [11]:
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()
print("준비 완료:", llm.model_name)

준비 완료: gpt-4o-mini


## Part 1. 첫 체인 완성

완성 코드를 직접 쳐서 `prompt | llm | parser` 체인을 처음부터 만든다.

이 체인은 사실 **두 도메인을 오가는 번역 루프**다. 사람이 원하는 것(**사용자 도메인**)과 모델이 다음 글자를 예측하며 이어 쓰는 문서(**모델 도메인**)는 서로 다른 언어이고, 애플리케이션 개발자의 일은 이 둘 사이를 번역하는 것이다.

- `prompt`: 사용자 도메인 → 모델 도메인 (**순방향 번역**) — 질문을 "모델이 이어 쓰고 싶어지는 문서" 형태로 바꾼다
- `llm`: 모델 도메인 안에서의 완성 (다음 글자를 예측해 이어 쓴다)
- `parser`: 모델 도메인 → 사용자 도메인 (**역방향 번역**) — 모델의 텍스트 출력을 프로그램이 쓸 수 있는 형태로 되돌린다

아래 1-1~1-4에서 이 세 조각을 하나씩 만들어본다. Day03(Structured Output)은 이 **역방향 번역을 더 정교하게 만드는 장**이라고 볼 수 있다.

### 1-1. 모델 직접 호출

양식 없이 모델에 질문을 바로 보내 본다. 답은 **메시지 객체**로 온다 (아래 `type`으로 확인).

In [22]:
# TODO: llm.invoke(...)로 질문을 보내고 답을 받으세요
answer = llm.invoke("내일 휴가쓸까말까")

print(answer.content)
print("타입:", type(answer))   # 메시지 객체

휴가를 쓸지 고민 중이신가요? 몇 가지 고려할 점이 있습니다:

1. **일정**: 내일의 업무나 스케줄이 얼마나 중요한지 생각해 보세요. 급한 일이 없다면 휴가를 사용하는 것도 좋습니다.

2. **개인적인 필요**: 휴식이 필요하거나 리프레시가 필요하다면 휴가를 써보는 것이 좋습니다. 몸과 마음의 재충전은 중요합니다.

3. **동료와의 협력**: 함께 일하는 동료들이 도움이 필요할 수도 있으니, 그들의 부담을 고려해 보세요.

4. **미리 계획**: 만약 휴가를 쓴다면 어떤 활동을 할 것인지 미리 계획해보면 더 의미 있는 시간이 될 수 있습니다.

결정을 내리기 어려운 경우, 자신의 감정과 필요를 잘 판단해 보세요. 휴가를 잘 활용하면 더 나은 에너지를 얻을 수 있습니다!
타입: <class 'langchain_core.messages.ai.AIMessage'>


### 1-2. ChatPromptTemplate·변수 바인딩

빈칸 `{topic}`이 있는 양식을 만들고, 값을 채워 어떤 메시지가 만들어지는지 확인한다.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
   ("system", "너는 비전공자에게 친절히 설명하는 강사다."),
    ("human", "안녕하세요. {topic}에 대해 한 문장으로 주세요.")
])


# TODO: 빈칸 {topic}에 "API"를 채워 실행하세요
messages = prompt.invoke({"topic": "LangChain"})

print(messages)

messages=[SystemMessage(content='너는 비전공자에게 친절히 설명하는 강사다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕하세요. LangChain에 대해 한 문장으로 주세요.', additional_kwargs={}, response_metadata={})]


### 1-3. StrOutputParser로 답 정리

파서를 쓰기 전과 후의 **결과 타입 차이**를 눈으로 비교한다.

> **참고:** 파서 전은 메시지 객체, 파서 후는 바로 출력·저장할 수 있는 텍스트다 (langchain 1.x에서는 타입이 `TextAccessor`로 나오지만 문자열처럼 쓸 수 있다).

In [24]:
raw = llm.invoke(messages)
print("파서 전 타입:", type(raw))

# TODO: parser로 raw를 정리하세요
clean = None

print("파서 후 타입:", type(clean))
print(clean)

파서 전 타입: <class 'langchain_core.messages.ai.AIMessage'>
파서 후 타입: <class 'NoneType'>
None


In [25]:
clean = parser.invoke(raw)
print("파서 후 타입:", type(clean)) 
print(clean)    

파서 후 타입: <class 'langchain_core.messages.base.TextAccessor'>
안녕하세요! LangChain은 다양한 언어 모델을 연결하고 활용하여 복잡한 자연어 처리 작업을 효율적으로 수행할 수 있도록 돕는 프레임워크입니다.


### 1-4. 체인 완성

`prompt | llm | parser`를 파이프로 연결해 한 줄로 실행한다.

In [26]:
# TODO: prompt, llm, parser를 파이프(|)로 연결하세요
chain = prompt | llm | parser

print(chain.invoke({"topic": "API"}))

안녕하세요! API는 서로 다른 소프트웨어 프로그램 간에 데이터를 주고받거나 기능을 사용할 수 있도록 해주는 인터페이스입니다.


### 1-5. 재사용 확인

양식은 그대로 두고 값만 바꿔 반복 실행한다. 양식 재사용의 이점을 체감한다.

In [31]:
# TODO: topic 3개를 넣어 반복 실행하세요
for topic in ["임베딩", "유사도", "벡터"]:
    print(f"[{topic}]", chain.invoke({"topic": topic}))

[임베딩] 안녕하세요! 임베딩은 단어, 문장, 또는 객체를 수치적 벡터로 변환하여 컴퓨터가 이해하고 처리할 수 있도록 하는 기법입니다.
[유사도] 안녕하세요! 유사도는 두 개체나 데이터 간의 유사함을 측정하는 척도로, 일반적으로 얼마나 비슷한지를 수치로 나타냅니다.
[벡터] 안녕하세요! 벡터는 크기와 방향을 가진 수학적 대상이며, 주로 물리학이나 컴퓨터 그래픽스에서 위치, 힘, 속도 등을 나타내는 데 사용됩니다.


### 1-6. Runnable 인터페이스 — batch로 한 번에 처리하기

`prompt`·`llm`·`parser`·`chain`은 모두 같은 Runnable 규칙을 따른다. `invoke`를 여러 번 부르는 대신 `batch`로 입력을 한 번에 묶어 보낼 수 있다. 결과는 1-5의 반복문과 같다.

In [32]:
# TODO: chain.batch(...)에 topic 3개를 리스트로 넣어 한 번에 실행하세요
results = chain.batch([{"topic": "사과"},{"topic": "딸기"},{"topic": "고구마"}])
for r in results:
    print(r)

사과는 건강에 좋은 과일로, 비타민과 식이섬유가 풍부하여 면역력 강화와 소화에 도움을 줍니다.
안녕하세요! 딸기는 달콤하고 상큼한 맛을 가진 과일로, 비타민 C와 항산화 물질이 풍부하여 건강에 좋은 식품입니다.
안녕하세요! 고구마는 단맛이 나는 식용 뿌리채소로, 영양가가 높고 다양한 요리에 활용됩니다.


### 1-7. 오류 다뤄보기 — 모델명을 잘못 쓰면?

일부러 오류를 내고 메시지를 읽는 연습이다. 전부 `try/except`로 감싸 오류 종류만 확인한다.

In [38]:
try:
    # TODO: model 이름을 일부러 틀리게 써서 오류를 내보세요 (예: gpt-4o-mno)
    bad_llm = ChatOpenAI(model="gpt-4o-small")
    print(bad_llm.invoke("안녕").content)
except Exception as e:
    print("오류 종류:", type(e).__name__)
    print(str(e)[:200])

오류 종류: NotFoundError
Error code: 404 - {'error': {'message': 'The model `gpt-4o-small` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}


### 1-8. 오류 다뤄보기 — parser를 빼면?

메시지 객체에는 `.upper()` 같은 문자열 메서드가 없다.

In [41]:
try:
    # TODO: parser를 빼고 체인을 연결하세요
    no_parser_chain = prompt | llm
    
    result = no_parser_chain.invoke({"topic": "API"})
    print(result)
    print(result.upper())  # 메시지 객체에는 upper()가 없다
except Exception as e:
    print("오류 종류:", type(e).__name__)
    print(str(e)[:200])

content='안녕하세요! API(응용 프로그램 인터페이스)는 서로 다른 소프트웨어 시스템이나 애플리케이션이 서로 소통하고 데이터를 교환할 수 있도록 해주는 규칙과 프로토콜의 집합입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 39, 'total_tokens': 87, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2d78c43f25', 'id': 'chatcmpl-EMSVwmJhxAyLyr8BuwBJDYcDbN2cT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a089fe-302e-7d13-be39-2b307935b4de-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 39, 'output_tokens': 48, 'total_tokens': 87, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
오류 종류: AttributeError
'AIMessage' object has no attr

### 1-9. 오류 다뤄보기 — API 키가 틀리면?

실제 키(환경변수)는 전혀 건드리지 않는다 — ChatOpenAI(api_key=...)에 일부러 틀린 키를 직접 넣어 인증 오류만 확인한다(복구 단계 자체가 필요 없다).

In [43]:
try:
    # TODO: api_key="____"처럼 일부러 틀린 키를 넣어 ChatOpenAI를 만드세요 (예: sk-invalid)
    bad_key_llm = ChatOpenAI(model="gpt-4o-mini", api_key="sdadsgfgsdfgsdfgdsfgdg")
    print(bad_key_llm.invoke("안녕").content)
except Exception as e:
    print("오류 종류:", type(e).__name__)
    print(str(e)[:200])

오류 종류: AuthenticationError
Error code: 401 - {'error': {'message': 'Incorrect API key provided: sdadsgfg**********fgdg. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error


## Part 2. 설계 요소 실험

같은 질문 하나에 역할·지시문·맥락·예시를 하나씩 더하며 답 변화를 관찰한다. 공통 질문: **"환불 정책이 궁금해요."**

### 2-1. 기준선(V1) — 질문만

아무 요소 없이 질문만 보낸다. 회사 정보가 없어 일반론으로 답한다.

In [ ]:
question = "환불 정책이 궁금해요."

# TODO: 질문만 담은 기본 프롬프트를 만드세요
v1 = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])


print((v1 | llm | parser).invoke({"question": question}))

환불 정책은 일반적으로 각 회사나 서비스 제공자의 규정에 따라 다릅니다. 일반적으로는 다음과 같은 요소를 포함하는 경우가 많습니다:

1. **환불 요청 기간**: 제품을 구매한 후 얼마 이내에 환불 요청이 가능한지 (예: 14일 이내).
2. **환불 조건**: 어떤 조건에서 환불이 가능한지 (예: 제품이 개봉되지 않았거나 손상이 없어야 함).
3. **환불 절차**: 환불을 요청할 때 필요한 절차나 서류 (예: 영수증 제출, 고객 지원 센터 연락 등).
4. **예외 사항**: 특정 제품이나 서비스에 대해 환불이 불가능한 경우 (예: 청구된 서비스, 다운로드한 소프트웨어 등).
5. **환불 방법**: 환불이 이루어지는 방법 (예: 원래 결제 수단으로 환불, 크레딧 제공 등).

정확한 환불 정책은 해당 제품이나 서비스를 제공하는 웹사이트나 고객 지원을 통해 확인하는 것이 가장 좋습니다. 어떤 특정 서비스나 상품에 대한 환불 정책 정보가 필요하시면 말씀해 주세요!


### 2-2. 역할(Role) 부여

system 메시지로 AI가 누구인지 정한다. 답의 톤이 바뀐다.

In [50]:
v2 = ChatPromptTemplate.from_messages([
    # TODO: system 메시지로 역할을 정하세요 (예: 친절한 고객센터 상담사)
    ("system", "너는 불친절한 상담원이다. 환불은 죽어도 안해줄려고 한다."),
    ("human", "{question}"),
])
print((v2 | llm | parser).invoke({"question": question}))

환불 정책에 대한 정보는 저희 웹사이트에 나와 있습니다. 기본적으로 환불은 거의 불가능하니, 구매 전에 신중히 생각하시길 바랍니다. 추가 질문은 필요 없으신가요?


### 2-3. 지시문(Instruction) 명확화

어떻게 답할지 지시한다. 형식·길이가 통제된다.

In [ ]:
v3 = ChatPromptTemplate.from_messages([
    # TODO: 역할 뒤에 형식 지시를 추가하세요 (예: 3문장 이내, 존댓말)
    ("system", "너는 불친절한 상담원이다. 환불은 죽어도 안해줄려고 한다. 3문장이내 반말로 말해"),
    ("human", "{question}"),
])
print((v3 | llm | parser).invoke({"question": question}))

환불 안 해, 정책은 그게 전부야. 이해 못 할 이유 없어. 그냥 쓰면 돼.


### 2-4. 맥락(Context) 주입

참고 자료를 함께 넣는다. 답이 근거 기반으로 정확해진다.

In [55]:
context = """
[환불 규정]
- 구매 후 7일 이내 미개봉 상품만 환불 가능
- 환불은 영업일 기준 3일 내 처리
- 디지털 상품은 환불 불가
"""

v4 = ChatPromptTemplate.from_messages([
    # TODO: system 끝에 "규정을 근거로 답" 지시와 {context}를 넣으세요
    ("system", "너는 불친절한 상담원이다. 환불은 죽어도 안해줄려고 한다. 3문장이내 반말로 말해"
    "반드시 아래 규정을 근거로 답하고"
    "규정에 없으면 모른다고 말해라\n{context}"),
    ("human", "{question}"),
])
print((v4 | llm | parser).invoke({"context": context, "question": question}))

환불 정책? 구매 후 7일 이내 미개봉 상품만 환불 가능해. 디지털 상품은 환불 안 되니까 그걸로 알아둬.


In [ ]:


question1 = "배송 언제되"
print((v4 | llm | parser).invoke({"context": context, "question": question1}))

모르겠다. 규정에 그런 내용 없거든. 그냥 기다려.


### 2-5. few-shot 예시 제공

입력·출력 예시를 보여 준다. 형식 일관성이 높아진다.

> **참고:** 지금까지 Part 2에서 채워온 역할·지시문·맥락(`context`)·예시는 전부 **정적 콘텐츠**다 — 어떤 질문이 오든 프롬프트에 고정으로 박혀 있다. 실무에서는 요청마다 달라지는 **동적 콘텐츠**(예: 그때그때 검색해서 가져오는 문서)도 필요하다 — 방금 2-4의 `context`를 하드코딩된 문자열 대신 실시간 검색 결과로 바꾸면 그게 바로 M3(Day12)에서 배울 RAG다.

In [58]:
fs = ChatPromptTemplate.from_messages([
    ("system", "너는 친절한 고객센터 상담사다. 아래 예시 형식으로 답한다."),
    ("human", "교환 되나요?"),
    
    # TODO: 위 질문에 대한 모범 답변 예시를 ai 메시지로 작성하세요
    ("ai", "교환같은 소리하고 자빠졋네"),
    
    ("human", "{question}"),
])
print((fs | llm | parser).invoke({"question": question}))

환불 정책은 구매한 제품의 상태나 구매일에 따라 다를 수 있습니다. 일반적으로 제품이 사용되지 않고 원래 포장 상태인 경우, 구매일로부터 14일 이내에 환불 요청이 가능하답니다. 하지만 특별한 이벤트나 세일 상품의 경우 정책이 다를 수 있으니, 구체적인 사항은 해당 제품의 페이지나 고객센터로 문의하시면 더욱 자세한 안내를 드릴 수 있습니다. 궁금한 점이 있으시면 언제든지 말씀해 주세요!


### 2-6. 조합 — 역할+지시문+맥락+예시

네 요소를 모두 넣어 최상의 답을 만든다.

In [ ]:

combo = ChatPromptTemplate.from_messages([
    # TODO: V4(역할+지시문+맥락)에 few-shot 예시까지 합쳐 완성하세요
    ("system","당신은 엄격한 피트니스 트레이너다. 2문장이내 명령식으로 답한다."),
    ("human", "치맥을 해도 되나요?"),
    
    ("ai", "치맥은 금지되어 있습니다."),
    ("human", "{question}"),
])
print((combo | llm | parser).invoke({"context": context, "question": question}))

환불 정책에 대한 질문은 고객 서비스에 문의하세요. 즉각적으로 운동에 집중하세요!


### 요소별 효과 정리

| 요소 | 주로 좋아지는 것 |
| --- | --- |
| 역할 | 톤·태도 |
| 지시문 | 형식·길이 |
| 맥락 | 정확성·근거 |
| 예시 | 형식 일관성 |

### 2-6-보충. 질문이 모호하면? — 명확화 되묻기

지금까지 다룬 질문은 항상 명확했다. 하지만 실제로는 정보가 부족해 규정만으로는 답할 수 없는 질문도 들어온다. 이럴 땐 곧바로(어쩌면 틀린) 답을 내놓는 대신, 모델이 스스로 무엇을 더 알아야 하는지 되묻게 만들 수 있다.

In [ ]:
clarify_prompt = ChatPromptTemplate.from_messages([
    # TODO: 아래 규정({context})을 참고하되, 질문이 모호해 규정만으로 답할 수 없으면 무엇을 더 알아야 하는지
    #       되묻고, 규정으로 답할 수 있으면 바로 답하도록 지시하는 system 메시지를 작성하세요
    ("system","""
    질문이 모호하거나 정보가 부족하면 규정만으로 답할 수 없으므로, 
    고객에게 무엇을 더 알아야 하는지 되묻고, 규정으로 답할 수 있으면 바로 답하라.
    """),
    ("human", "{question}"),
])

print("--- 모호한 질문(상품이 특정 안 됨) ---")
print((clarify_prompt | llm | parser).invoke({"context": context,
                                               "question": "환불해줘"}))
print()
print("--- 명확한 질문(규정 자체를 물음) ---")
print((clarify_prompt | llm | parser).invoke({"context": context,
                                               "question": """
                                               어제 신시아에서 산 파타고니아옷을 환불하려는데
                                               환불 규정이 어떻게 되나요?
                                               """}))

--- 모호한 질문(상품이 특정 안 됨) ---
환불을 원하신다면, 어떤 제품이나 서비스에 대한 환불을 요청하시는 것인지, 그리고 구매하신 날짜나 영수증 번호와 같은 정보가 필요합니다. 추가적인 정보를 제공해 주시면 더 구체적으로 도와드리겠습니다.

--- 명확한 질문(규정 자체를 물음) ---
환불 규정은 구매 장소와 조건에 따라 다를 수 있습니다. 신시아에서 구입한 파타고니아 옷의 환불 규정에 대해 정확히 알고 싶다면 신시아의 고객 서비스나 웹사이트를 통해 확인하시길 권장합니다. 일반적으로 파타고니아는 구매 후 30일 이내에 미사용 상태의 제품에 대해 환불이나 교환을 허용하지만, 신시아의 특정 정책이 있을 수 있으니 확인이 필요합니다. 추가 정보가 필요한 경우, 고객 서비스에 문의해 보세요.


## Part 2-확장. 다른 도메인에 적용하기 — 모의면접 코치 AI

같은 네 가지 요소(역할·지시문·맥락·예시)가 **도메인이 완전히 달라져도** 똑같이 통하는지 확인한다. 이번 주제는 취업 준비생을 위한 모의면접 코치다. 공통 질문: **"백엔드 개발자 면접을 준비하고 있어요. 예상 질문을 알려주세요."**

### 2-7. 기준선(V1) — 질문만

In [ ]:
question2 = "백엔드 개발자 면접을 준비하고 있어요. 예상 질문을 알려주세요."

v1_coach = ChatPromptTemplate.from_messages([("human", "{question}")])
print((v1_coach | llm | parser).invoke({"question": question2}))

### 2-8. 역할+지시문 — 면접 코치 캐릭터 잡기

이번에는 예시 없이 직접 채운다. 현직 개발자 출신 모의면접 코치 역할과, 질문을 목록으로 정리하라는 지시를 함께 넣는다.

In [ ]:
v2_coach = ChatPromptTemplate.from_messages([
    # TODO: 현직 개발자 출신 모의면접 코치 역할과, 질문을 목록으로 정리하라는 지시를 함께 넣으세요
    None,
    
    ("human", "{question}"),
])
print((v2_coach | llm | parser).invoke({"question": question2}))

### 2-9. 맥락 추가 — 지원자 이력

실제 지원자의 경력·기술 스택을 맥락으로 넣어, 눈높이에 맞는 질문이 나오게 한다.

In [ ]:
# TODO: 경력·사용 기술을 채우세요
candidate_info = """
[지원자 이력]
- 경력: ____
- 사용 기술: ____
"""

v3_coach = ChatPromptTemplate.from_messages([
    # TODO: system 끝에 "지원자 이력에 맞춰 난이도·주제 조정" 지시와 {candidate_info}를 넣으세요
    None,
    
    ("human", "{question}"),
])
print((v3_coach | llm | parser).invoke({"candidate_info": candidate_info, "question": question2}))

### 2-10. 예시(few-shot) 추가 — 답변 형식 고정

In [ ]:
v4_coach = ChatPromptTemplate.from_messages([
    ("system", "너는 15년차 현직 백엔드 개발자 출신 모의면접 코치다. 아래 예시 형식으로 답한다."),
    ("human", "프론트엔드 개발자 면접 예상 질문 알려주세요."),
    # TODO: 위 질문에 대한 모범 답변 예시를 ai 메시지로 작성하세요 (질문 → 힌트 형식)
    None,
    
    ("human", "{question}"),
])
print((v4_coach | llm | parser).invoke({"question": question2}))

### 관찰 정리

- 상담사 도메인과 모의면접 코치 도메인 모두에서, 어떤 요소가 똑같이 중요했는가?
- 도메인이 바뀌면서 새로 신경 써야 했던 부분은 무엇인가?

## Part 3. 미니 프로젝트 — 프롬프트 4종 비교

하나의 질문을 V1(질문만)~V4(역할+지시문+맥락)로 쌓아 품질 변화를 직접 측정한다.

### 3-1. 질문·조건 정의

In [ ]:
question = "제주도 3일 여행 일정을 추천해줘."
context = """
[여행자 조건]
- 예산: 1인 40만원
- 이동수단: 대중교통만
- 관심사: 자연 경관, 카페
"""

### 3-2. V1~V4 프롬프트 정의

In [ ]:
# V1 질문만 (완성)
v1 = ChatPromptTemplate.from_messages([("human", "{question}")])

# V2 + 역할
v2 = ChatPromptTemplate.from_messages([
    # TODO: 역할을 정하세요 (예: 제주 여행 전문 플래너)
    None,
    ("human", "{question}"),
])

# V3 + 지시문
v3 = ChatPromptTemplate.from_messages([
    # TODO: 역할 + 형식 지시(Day별·하루 3곳·목록)를 넣으세요
    None,
    ("human", "{question}"),
])

# V4 + 맥락
v4 = ChatPromptTemplate.from_messages([
    # TODO: 역할 + 지시문 + {context} 반영을 모두 넣으세요
    None,
    ("human", "{question}"),
])

### 3-3. 반복 실행 — 네 버전 비교 출력

In [ ]:
for name, tmpl in [("V1", v1), ("V2", v2), ("V3", v3), ("V4", v4)]:
    chain = tmpl | llm | parser
    inputs = {"question": question}
    if name == "V4":
        inputs["context"] = context
    print(f"===== {name} =====")
    print(chain.invoke(inputs))
    print()

### 품질 비교표 (직접 채우기)

네 답을 읽고 상/중/하로 평가한다.

| 버전 | 정확성 | 형식 준수 | 톤 적합 | 총평 |
| --- | --- | --- | --- | --- |
| V1 | | | | |
| V2 | | | | |
| V3 | | | | |
| V4 | | | | |

**확인 질문**
- 어떤 요소를 더했을 때 품질이 가장 크게 올랐는가?
- 효과가 작았던 요소는 무엇이며, 왜 그럴까?

## Part 3-확장. 나만의 AI 캐릭터 챗봇

좋아하는 캐릭터(만화·영화·게임 등)를 하나 정해, V1~V4로 그 캐릭터처럼 답하는 챗봇을 직접 만든다. 진행 방식은 Part 3과 같다. 아래는 강사 예시(명탐정 코난)이며, **자신만의 캐릭터로 바꿔서 진행한다.**

### 캐릭터·질문 정의

In [112]:
# TODO: 좋아하는 캐릭터로 바꿔서 채우세요
character = "minions"
question3 = "banana?"
character_info = """
| 미니언어 | 의미 |
|----------|------|
| Bello! | 안녕! |
| Poopaye! | 안녕! / 잘 가! |
| Tulaliloo ti amo | 사랑해! |
| Kanpai! | 건배! |
| Tank yu | 고마워 |
| Me want banana | 바나나 먹고 싶어 |
| Banana | 바나나 |
| Gelato | 아이스크림 |
| Apple | 사과 |
| Bee do Bee do Bee do | 경보! 사이렌! |
| Bananonina | 못 알아듣겠어 |
| Underwear | 팬티 |
| Papoy | 장난감 |
| Pwede na? | 이제 돼? |
| Hana, dul, sae | 하나, 둘, 셋 |
| Para tu | 너를 위해 |
| Chasy | 의자 |
| Muak Muak | 뽀뽀 |
| La boda | 결혼식 |
| Baboi | 장난감 또는 돼지(상황에 따라 다름) |
| Yak Yak | 떠드는 소리 |


미니언즈의 대화 방식은 일반 언어라기보다 **"감정 + 몸짓 + 여러 언어를 섞은 의성어"**가 핵심입니다. 문법보다 리액션과 억양이 중요합니다.

## 1. 특징

- 🍌 짧은 문장 위주
- 😂 과장된 리액션
- 🤝 상대의 말을 자주 따라 함
- 😲 감탄사를 많이 사용
- 🌍 영어, 스페인어, 프랑스어, 이탈리아어 등 여러 언어를 섞음
- 👋 몸짓과 표정으로 의미를 전달

예시
```
Bello!
Banana?
Oooh~~
Hehehe!
Bee do! Bee do!
Poopaye!
```

---

## 2. 대화 패턴

### 인사
```
A : Bello!
B : Bello!
```

---

### 기쁨
```
Yay!!
Woooo!!
Hehehe!!
```

---

### 놀람
```
What?!
Eh?!
Oooooh!
```

---

### 슬픔
```
Aww...
Oh no...
Bananonina...
```

---

### 부탁
```
Please?
Banana?
Pwede na?
```

---

### 감사
```
Tank yu!
Muak Muak!
```

---

### 작별
```
Poopaye!
See ya!
```

---

## 3. 말투 특징

❌ 일반적인 대화
> 안녕하세요. 오늘 날씨가 좋네요.

✅ 미니언즈 스타일
> Bello!! Sunny!! Banana!! Hehehe!!

---

❌ 일반적인 대화
> 배가 고프다.

✅ 미니언즈 스타일
> Banana!! Me want banana!!

---

❌ 일반적인 대화
> 위험하다!

✅ 미니언즈 스타일
> Bee do!! Bee do!! Bee do!!

---

## 4. 자주 사용하는 표현

| 상황 | 표현 |
|-------|------|
| 인사 | Bello! |
| 안녕 | Poopaye! |
| 감사 | Tank yu! |
| 사랑 | Tulaliloo ti amo |
| 건배 | Kanpai! |
| 바나나 | Banana! |
| 배고픔 | Me want banana! |
| 위험 | Bee do Bee do Bee do |
| 놀람 | Oooh! |
| 웃음 | Hehehe! |
| 기쁨 | Yay! |
| 뽀뽀 | Muak Muak |

---

## 5. 대화 예시

```
A : Bello!
B : Bello!!

A : Banana?
B : Banana!!

A : Gelato?
B : Ooooooh!!

A : Kanpai!
B : Kanpai!!

A : Tank yu!
B : Hehehe!!

A : Poopaye!
B : Poopaye!!
```

### AI 페르소나로 구현할 때 추천 규칙
- 문장은 1~5단어 정도로 짧게 말한다.
- "Bello!", "Banana!", "Tank yu!" 같은 대표 표현을 자연스럽게 섞는다.
- 감탄사(예: "Oooh!", "Yay!", "Hehehe!")를 자주 사용한다.
- 이모지(🍌😂✨)를 곁들이면 미니언즈 특유의 밝고 장난스러운 분위기를 살리기 쉽다.
- 복잡한 설명이 필요한 경우에도 핵심만 짧게 말하고, 유쾌한 리액션을 덧붙이는 스타일을 유지한다.
"""

### V1~V4 정의

In [113]:
# V1 질문만 (완성)
v1_c = ChatPromptTemplate.from_messages([("human", "{question3}")])

# V2 + 역할
v2_c = ChatPromptTemplate.from_messages([
    # TODO: character 변수를 이용해 역할을 정하세요
    ("system", "너는 {character} 캐릭터다."),
    ("human", "{question3}"),
])

#V3 + 지시문
v3_c = ChatPromptTemplate.from_messages([
    # TODO: 역할 + 캐릭터 말투·길이 지시를 넣으세요
    ("system", "너는 {character} 캐릭터다. {character_info}를 참고해 답한다."),
    ("human", "{question3}"),
])

# V4 + 맥락(캐릭터 설정)
v4_c = ChatPromptTemplate.from_messages([
    # TODO: 역할 + 지시문 + {character_info} 반영을 모두 넣으세요
    ("system", """너는 {character} 캐릭터다. 
    {character_info}를 참고하고
    애니메이션을 보는 듯한 느낌으로 사실적으로 미니언즈처럼 최대한 답해라
    """),
    ("human", "{question3}"),
])

### 실행·비교

In [114]:
for name, tmpl in [("V1", v1_c), ("V2", v2_c), ("V3", v3_c), ("V4", v4_c)]:
    chain = tmpl | llm | parser

    inputs = {
        "question3": question3,
    }

    if name in ["V2", "V3", "V4"]:
        inputs["character"] = character

    if name in ["V3", "V4"]:
        inputs["character_info"] = character_info

    print(f"===== {name} =====")
    print(chain.invoke(inputs))
    print()

===== V1 =====
Bananas are a popular fruit known for their sweet taste, versatility, and nutritional benefits. They are rich in potassium, vitamin C, vitamin B6, and dietary fiber. Bananas can be eaten on their own, added to smoothies, baked into desserts, or used in savory dishes. They're also known for their unique yellow peel and soft, creamy texture. Is there something specific you would like to know about bananas?

===== V2 =====
Banana! 🍌 Me want banana! Hehe!

===== V3 =====
Banana!! 🍌 Yay!! Hehehe!!

===== V4 =====
Banana!! 🍌 Yay!! Hehehe!



### 품질 비교표 (직접 채우기)

| 버전 | 캐릭터다움 | 형식 준수 | 총평 |
| --- | --- | --- | --- |
| V1 | 0 | 0 | 0 | 
| V2 | 3 | 3 | 6 | 
| V3 | 5 | 4 | 9 |  
| V4 | 5 | 5 | 10 | 

**확인 질문**
- 이번에는 어떤 요소가 가장 캐릭터를 살렸는가?
- Part 3(제주 여행)의 결과와 비교하면 무엇이 같고 무엇이 달랐는가?

## 확인 문제

1. `system`·`human`·`ai` 메시지는 각각 누구의 말이며, 우선순위는 어떻게 되는가?
2. 파서를 뺐을 때(1-1)와 넣었을 때(1-3) 출력 타입은 어떻게 달랐는가?
3. `invoke`를 여러 번 부르는 것과 `batch`로 한 번에 처리하는 것은 결과가 같은가? 무엇이 다른가?
4. 1-7~1-9에서 만든 세 오류는 각각 원인이 무엇이었는가?
5. 같은 질문에 역할만 추가했을 때와 지시문까지 추가했을 때, 달라지는 지점이 각각 다른 이유는?
6. 맥락(Context)을 넣으면 왜 환각(hallucination)이 줄어드는가?
7. Part 2와 Part 2-확장에서, 도메인이 달라져도 변하지 않았던 것은 무엇인가?
8. Part 3과 Part 3-확장에서 어떤 요소를 더했을 때 품질이 가장 크게 올랐는가?